# 16. 8/25 라벨 그룹 안 순서 개선
LB: submit_36(규칙 0.3) 0.6696, submit_38(규칙 0.5) 0.6707 → 규칙 비중 효과는 평평해짐.
남은 여지: **8/25 라벨=1 그룹 안에서 누구를 위로 두느냐.** 현재 규칙의 그룹 안 순서 = 과거 중앙값 낮은 순.
학습 쌍(전주 라벨 있는 행)에서 그룹별로 어떤 신호가 실제 Rising을 잘 올리는지 검증 (채널 GroupKFold OOF).

In [1]:
import pickle, numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
d = pickle.load(open('../data/processed/_refine_cache.pkl', 'rb'))
po, test, FC, INTER = d['po'], d['test'], d['FC'], d['INTER']
logreg = lambda C: Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                             ('m', LogisticRegression(max_iter=3000, C=C, class_weight='balanced'))])
# V1 OOF (채널 GroupKFold 5)
ch = po.channel_id.values; u = np.unique(ch); rng = np.random.default_rng(0); rng.shuffle(u)
fold = np.array([dict(zip(u, np.arange(len(u)) % 5))[c] for c in ch])
po['v1_oof'] = 0.0
for f in range(5):
    m = logreg(0.01).fit(po[fold != f][FC + INTER], po[fold != f].target)
    po.loc[fold == f, 'v1_oof'] = m.predict_proba(po[fold == f][FC + INTER])[:, 1]

cands = {'과거중앙값 낮은 순 (현재 규칙)': -po.log_past_med, 'V1 모델': po.v1_oof, '조건부 시뮬': po.sim_cond,
         'u_post (9월초 수준 추정)': po.u_post, 'c_surprise': po.c_surprise, 'pos_low': po.pos_low,
         'mom_7': po.mom_7}
def grp_ap(score, g):
    rows = po[(po.prev == g)]
    s = pd.Series(np.asarray(score), index=po.index).loc[rows.index]
    return np.mean([average_precision_score(r.target, s.loc[r.index].fillna(s.median())) for _, r in rows.groupby('ref_date') if r.target.sum() > 0])
rk = lambda x: pd.Series(np.asarray(x, float), index=po.index).groupby(po.ref_date).rank(pct=True)
res = {k: {'라벨=1 그룹': grp_ap(v, 1), '라벨=0 그룹': grp_ap(v, 0)} for k, v in cands.items()}
res['V1 + 과거중앙값 (순위평균)'] = {g: grp_ap(0.5 * rk(po.v1_oof) + 0.5 * rk(-po.log_past_med), x) for g, x in [('라벨=1 그룹', 1), ('라벨=0 그룹', 0)]}
res['V1 + 시뮬'] = {g: grp_ap(0.5 * rk(po.v1_oof) + 0.5 * rk(po.sim_cond), x) for g, x in [('라벨=1 그룹', 1), ('라벨=0 그룹', 0)]}
print('그룹 내 Rising 비율:', po.groupby(po.prev.fillna(-1)).target.mean().round(3).to_dict())
print(pd.DataFrame(res).T.round(4).sort_values('라벨=1 그룹', ascending=False))

그룹 내 Rising 비율: {-1.0: 0.217, 0.0: 0.109, 1.0: 0.503}
                    라벨=1 그룹  라벨=0 그룹
V1 모델                0.7927   0.2408
V1 + 시뮬              0.7580   0.2511
조건부 시뮬               0.7343   0.2456
u_post (9월초 수준 추정)   0.7197   0.2226
V1 + 과거중앙값 (순위평균)    0.7184   0.2344
c_surprise           0.6948   0.1023
pos_low              0.6485   0.1126
과거중앙값 낮은 순 (현재 규칙)   0.6482   0.1809
mom_7                0.5695   0.1137


### 결과: 라벨=1 그룹 안 PR-AUC — V1 모델 0.793 vs 과거중앙값 낮은 순(현재 규칙) 0.648. 라벨=0 그룹은 V1+시뮬 0.251이 최고.
## 제출 파일 (남은 2회)
- 모델 순위 M = 0.5·rank(submit_21) + 0.5·rank(V1)
- 그룹 안 순서 W = 라벨=1 그룹: V1 / 라벨=0 그룹: V1+시뮬 / 없음: M
- **submit_42** = 0.5·rank(라벨 + 1e-3·W) + 0.5·rank(M) (submit_38과 같은 그룹 비중, 그룹 안 순서만 교체)
- **submit_43** = 라벨=1 그룹 우선(라벨 없음은 0.2), 그룹 안은 M+W 순서

In [2]:
p_v1 = logreg(0.01).fit(po[FC + INTER], po.target).predict_proba(test[FC + INTER])[:, 1]
SUBD = '../submissions'
sample = pd.read_csv('../data/raw/epoch_data/sample_submission.csv', encoding='utf-8-sig')
t = test[['row_id', 'prev', 'sim_cond']].copy()
t['s21'] = pd.read_csv(f'{SUBD}/submit_21_ens_s4_s3_s15.csv').set_index('row_id').prediction.loc[t.row_id].values
t['v1'] = p_v1
N = len(t); r = lambda x: rankdata(pd.Series(np.asarray(x, float)).fillna(np.nanmedian(x))) / N
M = 0.5 * r(t.s21) + 0.5 * r(t.v1)
W = np.where(t.prev == 0, 0.5 * r(t.v1) + 0.5 * r(t.sim_cond), np.where(t.prev == 1, r(t.v1), M))
prev_f = t.prev.fillna(0.2).values

def save(score, fname):
    s = np.asarray(score, float) + 1e-7 * r(t.s21)
    s = rankdata(s) / N; s = (s - s.min()) / (s.max() - s.min())
    sub = sample[['row_id']].merge(pd.DataFrame({'row_id': t.row_id.values, 'prediction': s}), on='row_id', how='left')
    assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == 694 and sub.row_id.is_unique
    assert set(sub.row_id) == set(sample.row_id) and sub.prediction.between(0, 1).all() and sub.prediction.notna().all()
    assert not sub.prediction.duplicated().any()
    sub.to_csv(f'{SUBD}/{fname}', index=False, encoding='utf-8')
    return sub.set_index('row_id').prediction

s42 = save(0.5 * r(prev_f + 1e-3 * W) + 0.5 * M, 'submit_42_rule50_withinV1.csv')
s43 = save(prev_f + 0.5 * (0.5 * M + 0.5 * W), 'submit_43_prevgate_withinV1.csv')
s38 = pd.read_csv(f'{SUBD}/submit_38_ens21_prevrule50.csv').set_index('row_id').prediction
for nm, s in [('submit_42', s42), ('submit_43', s43)]:
    top = s.nlargest(139).index
    print(nm, '| submit_38과 순위상관', round(s.corr(s38.loc[s.index], method='spearman'), 4),
          '| 상위139 중 라벨=1', int((t.set_index('row_id').loc[top].prev == 1).sum()),
          '| 상위139가 38과 겹치는 수', len(set(top) & set(s38.nlargest(139).index)))

submit_42 | submit_38과 순위상관 0.8696 | 상위139 중 라벨=1 120 | 상위139가 38과 겹치는 수 128
submit_43 | submit_38과 순위상관 0.8715 | 상위139 중 라벨=1 134 | 상위139가 38과 겹치는 수 131
